# The Application of computational physics methods in finance -- Ali Khabazian // june 2024

1.3 Binomial option pricing model

In [41]:
import numpy as np

In [57]:
from functools import wraps
from time import time

def timing(f):
    @wraps(f)
    def wrappers(*args, **kw):
        ts = time()
        result = f(*args, **kw)
        te = time()
        print(f'Function {f.__name__} args: {args} took: {te-ts:.4f} seconds')
        return result
    return wrappers

In [58]:
#initial parameters

S0 = 100       #initial stock price
K = 100        #strike price
T = 1          #time to maturity in years
r = 0.06       #annual risk-free rate
N = 3          #number of time steps    
u = 1.1        #up-factor in binomial models  
d = 1/u        #ensure recombining tree
opttype = 'c'  #option type 'c'or 'p' 

Binomial tree slow

In [59]:
@timing
def binomial_tree_slow(K,T,S0,r,N,u,d,opttype='C'):
    
    #precompute constants
    dt = T/N
    q = (np.exp(r*dt) - d) / (u-d)
    disc = np.exp(-r*dt)

    #initialise asset prices at maturity - time step N
    S = np.zeros(N+1)
    S[0] = S0*d**N
    for j in range(1,N+1):
        S[j] = S[j-1]*u/d
    
    #initialise option values of maturity
    C = np.zeros(N+1)
    for j in range(0,N+1):
        C[j] = max(0,S[j]-K)

    #step backward through tree
    for i in np.arange(N,0,-1):
        for j in range(0,i):
            C[j] = disc * (q*C[j+1] +(1-q)*C[j])
 
    return C[0]

binomial_tree_slow(K,T,S0,r,N,u,d,opttype='C')

Function binomial_tree_slow args: (100, 1, 100, 0.06, 3, 1.1, 0.9090909090909091) took: 0.0000 seconds


10.145735799928817

Binomial fast tree

In [66]:
@timing
def binomial_tree_fast(K,T,S0,r,N,u,d,opttype='C'):
    
    #precompute constants
    dt = T/N
    q = (np.exp(r*dt) - d) / (u-d)
    disc = np.exp(-r*dt)

    #initialise asset prices at maturity - time step N
    C = S0 * d**(np.arange(N,-1,-1)) * u** (np.arange(0,N+1,1))
   
    #initialise option values of maturity
    C = np.maximum( C - K , np.zeros(N+1))

    #step backward through tree
    for i in np.arange(N,0,-1):
        C = disc * ( q * C[1:i+1] + (1-q) * C[0:i] )  # C[1:i+1] : up-vector and C[0:i] : down vector 
 
    return C[0]

binomial_tree_fast(K,T,S0,r,N,u,d,opttype='C')

Function binomial_tree_fast args: (100, 1, 100, 0.06, 3, 1.1, 0.9090909090909091) took: 0.0000 seconds


10.145735799928826

In [68]:
for N in [3, 50, 100, 1000, 5000]:
    binomial_tree_slow(K,T,S0,r,N,u,d,opttype='C')
    binomial_tree_fast(K,T,S0,r,N,u,d,opttype='C')

Function binomial_tree_slow args: (100, 1, 100, 0.06, 3, 1.1, 0.9090909090909091) took: 0.0000 seconds
Function binomial_tree_fast args: (100, 1, 100, 0.06, 3, 1.1, 0.9090909090909091) took: 0.0000 seconds
Function binomial_tree_slow args: (100, 1, 100, 0.06, 50, 1.1, 0.9090909090909091) took: 0.0008 seconds
Function binomial_tree_fast args: (100, 1, 100, 0.06, 50, 1.1, 0.9090909090909091) took: 0.0009 seconds
Function binomial_tree_slow args: (100, 1, 100, 0.06, 100, 1.1, 0.9090909090909091) took: 0.0030 seconds
Function binomial_tree_fast args: (100, 1, 100, 0.06, 100, 1.1, 0.9090909090909091) took: 0.0000 seconds
Function binomial_tree_slow args: (100, 1, 100, 0.06, 1000, 1.1, 0.9090909090909091) took: 0.3490 seconds
Function binomial_tree_fast args: (100, 1, 100, 0.06, 1000, 1.1, 0.9090909090909091) took: 0.0018 seconds
Function binomial_tree_slow args: (100, 1, 100, 0.06, 5000, 1.1, 0.9090909090909091) took: 6.0708 seconds
Function binomial_tree_fast args: (100, 1, 100, 0.06, 5000